Perfect 👍 Let’s extend the **Agent-to-Agent LangGraph example** with **conditional edges** so that the **Order Agent only runs if weather is good**, otherwise the **Cancel Agent runs**.

---

## 📝 Example: `agent2agent_conditional.py`

```python
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from typing import Annotated, TypedDict


# ---- Define State ----
class State(TypedDict):
    messages: Annotated[list, add_messages]
    weather: str  # store weather status


# ---- Define Tools ----
@tool
def get_weather(location: str) -> str:
    """Check if weather is good or bad in a location."""
    if location.lower() == "mumbai":
        return "bad"   # simulate rainy day
    return "good"


@tool
def place_order(item: str, qty: int) -> str:
    """Place an order for items."""
    return f"✅ Order placed: {qty} x {item}"


@tool
def cancel_order(item: str) -> str:
    """Cancel an order."""
    return f"❌ Order cancelled for {item} due to bad weather"


# ---- Initialize LLMs ----
weather_llm = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools([get_weather])
order_llm = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools([place_order])
cancel_llm = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools([cancel_order])


# ---- Agent Nodes ----
def weather_agent(state: State):
    msg = weather_llm.invoke(state["messages"])
    # Save weather info for decision making
    state["weather"] = msg.content.strip().lower()
    return {"messages": [msg], "weather": state["weather"]}


def order_agent(state: State):
    msg = order_llm.invoke(state["messages"])
    return {"messages": [msg]}


def cancel_agent(state: State):
    msg = cancel_llm.invoke(state["messages"])
    return {"messages": [msg]}


# ---- Condition Function ----
def route_based_on_weather(state: State) -> str:
    """Decide next agent based on weather status"""
    if state.get("weather") == "good":
        return "order_agent"
    return "cancel_agent"


# ---- Build Graph ----
builder = StateGraph(State)

builder.add_node("weather_agent", weather_agent)
builder.add_node("order_agent", order_agent)
builder.add_node("cancel_agent", cancel_agent)

builder.add_edge(START, "weather_agent")

# Conditional edge from weather_agent
builder.add_conditional_edges("weather_agent", route_based_on_weather, ["order_agent", "cancel_agent"])

builder.add_edge("order_agent", END)
builder.add_edge("cancel_agent", END)

graph = builder.compile()


# ---- Run Example ----
if __name__ == "__main__":
    user_input = {"messages": [{"role": "user", "content": "Order 10 umbrellas for Mumbai"}]}
    result = graph.invoke(user_input)

    print("\n=== Conversation ===")
    for m in result["messages"]:
        print(f"{m.type.upper()}: {m.content}")

    print("\nFinal Weather Decision:", result.get("weather"))
```

---

### 🔄 Flow

1. **User**: *“Order 10 umbrellas for Mumbai”*
2. **Weather Agent** → Calls `get_weather` → returns **"bad"**.
3. `route_based_on_weather` function decides → route to **Cancel Agent**.
4. **Cancel Agent** → *❌ “Order cancelled due to bad weather”*.

If user had asked for *Delhi*, flow would route to **Order Agent** ✅.

---

👉 This is a **robust agent-to-agent pipeline** with **conditional routing** in LangGraph.

Do you want me to also show how to **visualize this flow as a graph (Mermaid or PNG)** so you can see the routing clearly in VS Code?
